In [ ]:
!pip install music21
!pip install mido
!pip install pyfluidsynth

In [ ]:
# Install fluidsynth and a general MIDI soundfont
!apt-get update && apt-get install -y fluidsynth fluid-soundfont-gm

# Install ffmpeg for MP3 conversion
!apt-get install -y ffmpeg

In [ ]:
#collecting data

import os
from music21 import corpus, midi

def collect_bach_chorales(num_files=20, output_dir='midi_data'):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Collecting {num_files} Bach chorales...")
    # Get list of Bach chorales in the corpus
    bach_bundle = corpus.search('bach', 'composer')

    count = 0
    for result in bach_bundle:
        if count >= num_files:
            break
        try:
            score = result.parse()
            # Save as MIDI
            mf = midi.translate.music21ObjectToMidiFile(score)
            filename = os.path.join(output_dir, f"bach_{count}.mid")
            mf.open(filename, 'wb')
            mf.write()
            mf.close()
            print(f"Saved {filename}")
            count += 1
        except Exception as e:
            print(f"Error processing {result}: {e}")

if __name__ == "__main__":
    collect_bach_chorales()


In [ ]:
#preproccesing

import os
import pickle
import numpy as np
from music21 import converter, instrument, note, chord
from tensorflow.keras.utils import to_categorical

# Define directories
MIDI_DATA_DIR = 'midi_data'
PROCESSED_DATA_DIR = 'data'

def get_notes(midi_dir=MIDI_DATA_DIR):
    """Extract notes and chords from MIDI files."""
    notes = []
    if not os.path.exists(midi_dir):
        print(f"MIDI data directory '{midi_dir}' not found. Please ensure MIDI files are collected first.")
        return []

    for file in os.listdir(midi_dir):
        if file.endswith(".mid"):
            print(f"Parsing {file}...")
            midi_path = os.path.join(midi_dir, file)
            try:
                midi_obj = converter.parse(midi_path)
                notes_to_parse = None

                # Try to get parts
                parts = instrument.partitionByInstrument(midi_obj)
                if parts: # file has instrument parts
                    notes_to_parse = parts.parts[0].recurse()
                else: # file has notes in a flat structure
                    notes_to_parse = midi_obj.flat.notes

                for element in notes_to_parse:
                    if isinstance(element, note.Note):
                        notes.append(str(element.pitch))
                    elif isinstance(element, chord.Chord):
                        notes.append(".".join(str(n) for n in element.normalOrder))
            except Exception as e:
                print(f"Error parsing {file}: {e}")

    return notes

def prepare_sequences(notes, n_vocab, sequence_length=100):
    """Prepare the sequences used by the Neural Network."""
    if not notes:
        print("No notes found to prepare sequences.")
        return np.array([]), np.array([]), [], 0

    # Get all pitch names
    pitchnames = sorted(set(item for item in notes))

    # Create a dictionary to map pitches to integers
    note_to_int = dict((note, number) for number, note in enumerate(pitchnames))

    network_input = []
    network_output = []

    # Create input sequences and the corresponding outputs
    for i in range(0, len(notes) - sequence_length, 1):
        sequence_in = notes[i:i + sequence_length]
        sequence_out = notes[i + sequence_length]
        network_input.append([note_to_int[char] for char in sequence_in])
        network_output.append(note_to_int[sequence_out])

    n_patterns = len(network_input)

    if n_patterns == 0:
        print("Not enough data to create sequences of specified length.")
        return np.array([]), np.array([]), pitchnames, n_vocab

    # Reshape the input into a format compatible with LSTM layers
    network_input = np.reshape(network_input, (n_patterns, sequence_length, 1))
    # Normalize input
    network_input = network_input / float(n_vocab)

    # One-hot encode the output
    network_output = to_categorical(network_output, num_classes=n_vocab)

    return network_input, network_output, pitchnames

if __name__ == "__main__":
    # Ensure the processed data directory exists
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

    notes = get_notes()
    if not notes:
        print("No notes were extracted. Exiting preprocessing.")
    else:
        n_vocab = len(set(notes))
        network_input, network_output, pitchnames = prepare_sequences(notes, n_vocab)

        if network_input.size > 0:
            # Save preprocessed data
            with open(os.path.join(PROCESSED_DATA_DIR, 'notes.pkl'), 'wb') as f:
                pickle.dump(notes, f)

            np.save(os.path.join(PROCESSED_DATA_DIR, 'network_input.npy'), network_input)
            np.save(os.path.join(PROCESSED_DATA_DIR, 'network_output.npy'), network_output)
            with open(os.path.join(PROCESSED_DATA_DIR, 'pitchnames.pkl'), 'wb') as f:
                pickle.dump(pitchnames, f)

            print(f"Preprocessing complete. Vocab size: {n_vocab}, Patterns: {len(network_input)}")
        else:
            print("No sequences were generated. Check data or sequence length.")


In [ ]:
#Model training

import numpy as np
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Activation
from tensorflow.keras.callbacks import ModelCheckpoint

def create_model(network_input, n_vocab):
    """Create the structure of the neural network."""
    model = Sequential()
    model.add(LSTM(
        256,
        input_shape=(network_input.shape[1], network_input.shape[2]),
        return_sequences=True
    ))
    model.add(Dropout(0.3))
    model.add(LSTM(256))
    model.add(Dropout(0.3))
    model.add(Dense(256))
    model.add(Dropout(0.3))
    model.add(Dense(n_vocab))
    model.add(Activation('softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='rmsprop')

    return model

def train():
    """Train the neural network."""
    # Load preprocessed data
    network_input = np.load('data/network_input.npy')
    network_output = np.load('data/network_output.npy')
    with open('data/pitchnames.pkl', 'rb') as f:
        pitchnames = pickle.load(f)

    n_vocab = len(pitchnames)

    model = create_model(network_input, n_vocab)

    # Checkpoint to save the best weights
    filepath = "weights-improvement-{epoch:02d}-{loss:.4f}-bigger.keras"
    checkpoint = ModelCheckpoint(
        filepath,
        monitor='loss',
        verbose=0,
        save_best_only=True,
        mode='min'
    )
    callbacks_list = [checkpoint]

    print("Starting training...")
    model.fit(network_input, network_output, epochs=50, batch_size=64, callbacks=callbacks_list)

    # Save the final model
    model.save('music_generation_model.keras')
    print("Training complete. Model saved.")

if __name__ == "__main__":
    train()


In [ ]:
#generate music

import pickle
import numpy as np
from tensorflow.keras.models import load_model
from music21 import instrument, note, stream, chord

def generate():
    """Generate a piano midi file."""
    # Load the notes and pitchnames
    with open('data/notes.pkl', 'rb') as f:
        notes = pickle.load(f)
    with open('data/pitchnames.pkl', 'rb') as f:
        pitchnames = pickle.load(f)

    n_vocab = len(pitchnames)

    # Load the model
    model = load_model('music_generation_model.keras')

    # Prepare input sequences
    sequence_length = 100
    note_to_int = dict((note, number) for number, note in enumerate(pitchnames))

    network_input = []
    for i in range(0, len(notes) - sequence_length, 1):
        sequence_in = notes[i:i + sequence_length]
        network_input.append([note_to_int[char] for char in sequence_in])

    # Pick a random sequence from the input as a starting point
    start = np.random.randint(0, len(network_input)-1)
    int_to_note = dict((number, note) for number, note in enumerate(pitchnames))

    pattern = network_input[start]
    prediction_output = []

    # Generate 200 notes
    print("Generating notes...")
    for note_index in range(200):
        prediction_input = np.reshape(pattern, (1, len(pattern), 1))
        prediction_input = prediction_input / float(n_vocab)

        prediction = model.predict(prediction_input, verbose=0)

        index = np.argmax(prediction)
        result = int_to_note[index]
        prediction_output.append(result)

        pattern.append(index)
        pattern = pattern[1:len(pattern)]

    create_midi(prediction_output)

def create_midi(prediction_output):
    """Convert the output from the prediction to notes and create a midi file."""
    offset = 0
    output_notes = []

    # Create note and chord objects based on the values generated by the model
    for pattern in prediction_output:
        # Pattern is a chord
        if ('.' in pattern) or pattern.isdigit():
            notes_in_chord = pattern.split('.')
            notes = []
            for current_note in notes_in_chord:
                new_note = note.Note(int(current_note))
                new_note.storedInstrument = instrument.Piano()
                notes.append(new_note)
            new_chord = chord.Chord(notes)
            new_chord.offset = offset
            output_notes.append(new_chord)
        # Pattern is a note
        else:
            new_note = note.Note(pattern)
            new_note.offset = offset
            new_note.storedInstrument = instrument.Piano()
            output_notes.append(new_note)

        # Increase offset each iteration so that notes do not stack
        offset += 0.5

    midi_stream = stream.Stream(output_notes)
    midi_stream.write('midi', fp='generated_music.mid')
    print("Generated music saved to generated_music.mid")

if __name__ == "__main__":
    generate()



In [ ]:
# Replace 'generated_music.mid' with the actual name of your MIDI file
midi_file = 'generated_music.mid'
wav_file = 'generated_music.wav'
soundfont = '/usr/share/sounds/sf2/FluidR3_GM.sf2'

!fluidsynth -ni {soundfont} {midi_file} -F {wav_file} -r 44100
print(f"Successfully converted {midi_file} to {wav_file}")